# visualization_005

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (146).ipynb`

| Field | Value |
|---|---|
| Section | `visualization` |
| Market | `unknown` |
| Trading style | `scalping` |
| Timeframe | `unknown` |
| Code cells | 13 |
| Detected functions | next_year_prices, append_future_prices, get_price |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd

data = pd.read_csv('/content/drive/MyDrive/ipynb/Nat_Gas.csv')
data.head()

In [ ]:
import plotly.express as px
import pandas as pd

# Create the plot
fig = px.line(data, x='Dates', y='Prices', title="Prices over Dates")

# Update layout to rotate x-axis tick labels
fig.update_layout(
    xaxis=dict(
        tickangle=24  # Rotate ticks to 24 degrees
    ),
    xaxis_title="Dates",
    yaxis_title="Prices"
)

# Show the plot
fig.show()


In [ ]:
import pandas as pd

# Convert 'Dates' to datetime and set as index
data['Dates'] = pd.to_datetime(data['Dates'])

data['Year'] = data['Dates'].dt.year
data['Month'] = data['Dates'].dt.month

data.tail()

In [ ]:
from sklearn.linear_model import LinearRegression
import numpy as np

X = np.array(data[data['Month'] == 1]['Year']).reshape(-1, 1)
Y = np.array(data[data['Month'] == 1]['Prices'])

model = LinearRegression().fit(X, Y)

In [ ]:
model.predict(X)

In [ ]:
def next_year_prices(next_year):
    price_list = []
    for i in np.arange(12):
        # Extract the feature and target for the current month
        X = np.array(data[data['Month'] == 1+i]['Year']).reshape(-1, 1)
        Y = np.array(data[data['Month'] == 1+i]['Prices'])

        # Fit the linear regression model
        model = LinearRegression().fit(X, Y)

        # Predict the price for the next year
        price = model.predict([[next_year]])
        price_list.append(price[0])  # Append the scalar value, not array

    return price_list


In [ ]:
price25 = next_year_prices(2025)
np.array(price25)

In [ ]:
# Assuming 'data' is your main DataFrame with columns 'Year', 'Month', 'Prices'
def append_future_prices(data, next_year):
    # Generate predictions for the next year
    future_prices = next_year_prices(next_year)

    # Create a DataFrame for the next year's predictions
    future_data = pd.DataFrame({
        'Year': [next_year] * 12,  # Repeated next_year for each month
        'Month': list(range(1, 13)),  # Months from 1 to 12
        'Prices': future_prices  # Predicted prices
    })

    # Concatenate the original data with the future data
    updated_data = pd.concat([data, future_data], ignore_index=True)

    return updated_data

# Example usage
next_year = 2025
updated_data = append_future_prices(data, next_year)
print(updated_data.tail())


In [ ]:
import pandas as pd

# Convert Year and Month to Datetime, setting day as the last day of the month
updated_data['Datetime'] = pd.to_datetime(updated_data[['Year', 'Month']].assign(Day=1)) + pd.offsets.MonthEnd(0)

print(updated_data.tail())


In [ ]:
updated_data = updated_data.drop(columns=['Dates'], axis = -1)
updated_data

In [ ]:
import matplotlib.pyplot as plt

plt.plot(updated_data['Datetime'], updated_data['Prices'])
plt.show()

In [ ]:
def get_price(month, year):
  print(updated_data[(updated_data['Month'] == month) & (updated_data['Year'] == year)]['Prices'])

In [ ]:
get_price(11, 2025)